In [1]:
import pandas as pd
import re
pattern=r"(?:January|February|March|April|May|June|July|August|September|October|November|December)-\d{1,2}-\d{4}"

In [9]:
seasons=[
    '2019-20',
    '2020-21',
    '2021-22',
    '2022-23',
    '2023-24',
    '2024-25',
]
data_path='../../data/raw/premier_league/'

In [3]:
target_columns=[
    'player',
    'performance_ast',
    'performance_gls',
    'performance_sh',
    'performance_sot',
    'performance_crdy',
    'corner_kicks_in',
    'corner_kicks_out',
    'corner_kicks_str',
    'player_ast',
]

# Match Data Processing

In [4]:
def process_match_target_var(home_df, away_df, match_df, match_name):
    assert 'Players' in home_df.iloc[0, 0]
    assert 'Players' in away_df.iloc[0, 0]
    home_ind=home_df.iloc[0, 0]
    away_ind=away_df.iloc[0, 0]
    home_df=home_df.set_index('player')
    away_df=away_df.set_index('player')
    targets={
        'home_goals': home_df.loc[home_ind, 'performance_gls'],
        'away_goals': away_df.loc[away_ind, 'performance_gls'],
        'home_corners': home_df.loc[home_ind, 'corner_kicks_in']+home_df.loc[home_ind, 'corner_kicks_out']+home_df.loc[home_ind, 'corner_kicks_str'],
        'away_corners': away_df.loc[away_ind, 'corner_kicks_in']+away_df.loc[away_ind, 'corner_kicks_out']+away_df.loc[away_ind, 'corner_kicks_str'],
        'home_cards': home_df.loc[home_ind, 'performance_crdy'],
        'away_cards': away_df.loc[away_ind, 'performance_crdy'],
        'home_shots': home_df.loc[home_ind, 'performance_sh'],
        'away_shots': away_df.loc[away_ind, 'performance_sh'],
        'home_sots': home_df.loc[home_ind, 'performance_sot'],
        'away_sots': away_df.loc[away_ind, 'performance_sot'],
    }
    targets_df=pd.DataFrame(pd.Series(targets)).transpose()
    targets_df['home']=match_df.index.tolist()[0]
    targets_df['away']=match_df.index.tolist()[1]
    match_date=re.findall(pattern, match_name, re.IGNORECASE)[0]
    targets_df['date']=[match_date]
    return targets_df

# Generate all target data for all matches

In [10]:
for season in seasons:
    season_path=f'{data_path}/{season}/'
    all_matches=pd.read_csv(f'{season_path}/all_matches.csv').iloc[:,1].tolist()
    target_df=[]
    for match_name in all_matches:
        home_df=pd.read_csv(f'{season_path}/{match_name}_home_player_stat.csv')
        away_df=pd.read_csv(f'{season_path}/{match_name}_away_player_stat.csv')
        match_df=pd.read_csv(f'{season_path}/{match_name}_match_stat.csv', index_col=0)
        target_df.append(process_match_target_var(home_df, away_df, match_df, match_name))
    target_df=pd.concat(target_df, axis=0)
    #target_df.to_csv(f'{season_path}/all_target_df.csv', index=False)
    

In [16]:
target_df

,home_goals,away_goals,home_corners,away_corners,home_cards,away_cards,home_shots,away_shots,home_sots,away_sots,home,away,date
0,2,0,7,5,4,4,9,20,3,6,Newcastle Utd,Tottenham,September-1-2024
0,4,1,4,1,2,3,13,1,4,1,Manchester City,Ipswich Town,August-24-2024
0,0,1,6,2,4,4,14,5,5,3,Liverpool,Nott'ham Forest,September-14-2024
0,1,1,3,2,2,3,21,11,5,3,Fulham,West Ham,September-14-2024
0,1,0,5,8,2,3,14,10,5,2,Manchester Utd,Fulham,August-16-2024
0,2,0,6,2,2,2,18,9,6,3,Arsenal,Wolves,August-17-2024
0,1,2,5,3,1,2,13,15,2,3,West Ham,Aston Villa,August-17-2024
0,4,0,7,5,0,0,13,10,7,1,Tottenham,Everton,August-24-2024
0,0,3,6,3,5,2,15,12,7,5,West Ham,Chelsea,September-21-2024
0,1,1,4,3,4,2,13,9,7,3,Chelsea,Crystal Palace,September-1-2024


# Other Statistics

In [12]:
def process_match_other_var(home_df, away_df, match_df, match_name):
    assert 'Players' in home_df.iloc[0, 0]
    assert 'Players' in away_df.iloc[0, 0]
    home_df=home_df.drop(columns=target_columns).rename(columns=lambda x: f'home_{x}').head(1)
    away_df=away_df.drop(columns=target_columns).rename(columns=lambda x: f'away_{x}').head(1)
    data_df=pd.concat([home_df, away_df], axis=1)
    data_df['home']=[match_df.index.tolist()[0]]
    data_df['away']=[match_df.index.tolist()[1]]
    match_date=re.findall(pattern, match_name, re.IGNORECASE)[0]
    data_df['date']=[match_date]
    return data_df

In [13]:
for season in seasons:
    season_path=f'{data_path}/{season}/'
    all_matches=pd.read_csv(f'{season_path}/all_matches.csv').iloc[:,1].tolist()
    data_df=[]
    for match_name in all_matches:
        home_df=pd.read_csv(f'{season_path}/{match_name}_home_player_stat.csv')
        away_df=pd.read_csv(f'{season_path}/{match_name}_away_player_stat.csv')
        match_df=pd.read_csv(f'{season_path}/{match_name}_match_stat.csv', index_col=0)
        data_df.append(process_match_other_var(home_df, away_df, match_df, match_name))
    data_df=pd.concat(data_df, axis=0)
    #data_df.to_csv(f'{season_path}/all_data_df.csv', index=False)